In [ ]:
# Colab setup: fetch the data and the helper file (skip if you run locally with the files here)
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/osterild_fast_uvw.csv -O osterild_fast_uvw.csv
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/l6_helpers.py -O l6_helpers.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from l6_helpers import *

case = "moderate"                       # "strong" in the bonus
t, d = load_hour(case)                  # d["u"], d["v"], d["w"]: one hour, each 10-min block demeaned
U = U_MEAN[case]
print(f"{case}: U = {U} m/s, {t.size} samples = {t.size * DT / 60:.0f} min; "
      f"variances u, v, w = {np.var(d['u']):.3f}, {np.var(d['v']):.3f}, {np.var(d['w']):.3f} m²/s²")

# Round 2 — Read the real spectrum
### 46100 Micrometeorology · Lecture 6

In Round 1 you built the analyser. Now you turn it into a **spectrum estimator** exactly as in the lecture notes
(eq. 4.3.15–4.3.16), and use it on a full hour of $u$, $v$ and $w$ to find the −5/3 range and test the 4/3 law.

Convention (notes, eq. 4.3.1–4.3.3): $S(\omega)$ is **two-sided**, in rad/s: $\sigma^2 = \int_{-\infty}^{\infty} S\,d\omega = 2\int_0^\infty S\,d\omega$.

**Time: ~30' work + 5' pair answers.**

| Task | Time | What you'll do |
|---|---|---|
| 1 | 8' | Write `spectrum(x, dt, K)` and pass the Parseval check |
| 2 | 5' | How many blocks K? |
| 3 | 9' | The −5/3 range, in rad/s and in metres |
| 4 | 8' | The 4/3 law with $v$ and $w$ |

> **Getting stuck is part of this.** These tasks are designed so that you work it out; nobody expects you to
> know the answer beforehand. Stuck for 3 minutes? Open hint 1. **Call the teacher or TA** when you've opened two hints,
> or a check FAILS and you can't see why. Tell us what you expected and what you got.

## Task 1 — Your spectrum estimator (8')
Write `spectrum(x, dt, K)`:
- split `x` into K equal blocks (remove each block's mean),
- for each block, apply eq. 4.3.15 using the FFT,
- average the K results (eq. 4.3.16),
- return `omega` (rad/s, from 0 to the Nyquist frequency) and `S` at those frequencies.

Keep `S` two-sided, as in the notes. The Parseval check must PASS (it doubles the area for you).

In [ ]:
def spectrum(x, dt, K=1):
    # your code here
    omega, S = None, None
    return omega, S

omega, S_u = spectrum(d["u"], DT, K=6)
assert omega is not None and S_u is not None, "Write spectrum() first."
target = np.mean([b.var() for b in np.split(d["u"], 6)])        # variance seen by 10-min blocks
check_parseval(omega, S_u, target, "u, K = 6")

<details><summary><b>Hint 1</b></summary>

Round 1: $X(\omega_l,T)$ = `X_fft * dt / (2*np.pi)` and $S(\omega_l,T) = \frac{2\pi}{T}XX^*$ (eq. 4.3.9), with $T = N\Delta t$ for one block. Combine the two.

</details>
<details><summary><b>Hint 2</b></summary>

Combined: $S = \frac{\Delta t}{2\pi N}|X_{FFT}|^2$, which is eq. 4.3.15. `np.split(x, K)` gives the blocks; `np.fft.rfft` keeps only $\omega \ge 0$; `np.fft.rfftfreq(N, dt)` gives frequencies in Hz (multiply by 2π).

</details>
<details><summary><b>Hint 3</b></summary>

For each block: `np.abs(np.fft.rfft(block))**2 * dt / (2*np.pi*N)`, then average over blocks. `omega = 2*np.pi*np.fft.rfftfreq(N, dt)`.

</details>

*Compare your one line with eq. 4.3.15 in the notes: $\frac{1}{2\pi N f_s}\,|\sum_n x(n\Delta t)\,e^{-i2\pi l n/N}|^2$. Same thing?*

## Task 2 — How many blocks? (5')
**Commit first:** with more blocks (K = 1 → 6 → 36), what happens to (a) the scatter of the curve and (b) the lowest frequency it shows?
Then run the cell (it uses your function) and compare with Figure 4.3.3 in the notes.

In [ ]:
my_guess_K = None      # e.g. "more blocks: less scatter, and the lowest frequency goes ..."

plt.figure(figsize=(8, 4.5))
for K, sh in [(1, 1), (6, 10), (36, 100)]:
    om_, S_ = spectrum(d["u"], DT, K=K)
    plt.loglog(om_[1:], sh * om_[1:] * S_[1:], lw=0.6, label=f"K = {K}  (shifted ×{sh})")
plt.xlabel("ω (rad/s)"); plt.ylabel("ω S(ω)  (shifted)"); plt.legend(); plt.show()
print("your guess:", my_guess_K)

*Your conclusion (one sentence):* …

## Task 3 — The −5/3 range, in rad/s and in metres (9')
**Commit first:** the slope of the straight part at high frequency.
1. Choose `lo`, `hi` (rad/s) bracketing the straight part; the cell fits the slope (K = 6 spectrum, no further smoothing).
2. Write the eddy sizes $\lambda$ at the two ends (Taylor: $\omega = Uk = 2\pi U/\lambda$).
The right-hand plot is the book's premultiplied $\omega S(\omega)$ (Fig. 4.3.2): equal areas, equal variance.

In [ ]:
my_guess_slope = None

# your code here: the range of the straight part (rad/s)
lo, hi = None, None
assert None not in (lo, hi, my_guess_slope), "Commit a slope and choose lo, hi first."

slope = fit_slope(omega[1:], S_u[1:], lo, hi)
compare("slope", my_guess_slope, slope)

# your code here: eddy sizes (m) at the two ends of your range
lam_small, lam_big = None, None
assert None not in (lam_small, lam_big), "Write lam_small and lam_big first."
print(f"the straight part covers eddies from {lam_small:.1f} m to {lam_big:.0f} m")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].loglog(omega[1:], S_u[1:], lw=0.5)
m = (omega >= lo) & (omega <= hi); ref = np.median(S_u[m] * omega[m] ** (5 / 3)) * omega[m] ** (-5 / 3)
ax[0].loglog(omega[m], ref, "k--", lw=2, label="−5/3"); ax[0].legend(); ax[0].set_xlabel("ω (rad/s)"); ax[0].set_ylabel("S_u(ω)")
ax[1].semilogx(omega[1:], omega[1:] * S_u[1:], lw=0.5); ax[1].set_xlabel("ω (rad/s)"); ax[1].set_ylabel("ω S_u(ω)  (m²/s²)")
plt.tight_layout(); plt.show()

<details><summary><b>Hint 1</b></summary>

The highest frequency in your range belongs to the smallest eddies.

</details>
<details><summary><b>Hint 2</b></summary>

From $\omega = 2\pi U/\lambda$: $\lambda = 2\pi U/\omega$.

</details>
<details><summary><b>Hint 3</b></summary>

`lam_small, lam_big = 2*np.pi*U/hi, 2*np.pi*U/lo`

</details>

## Task 4 — The 4/3 law (8')
In the inertial range, local isotropy predicts $S_v = S_w = \tfrac43 S_u$ (a **ratio** at the same frequency, not a slope).
**Commit first:** at low or at high frequency, which ratio will reach 4/3 first, $S_v/S_u$ or $S_w/S_u$? Why?
Write the two spectra (same K as for $u$), then run.

In [ ]:
my_guess_43 = None

# your code here: the spectra of v and w
S_v, S_w = None, None
assert S_v is not None and S_w is not None, "Write S_v and S_w first."

bands = [(0.3, 1), (1, 5), (5, 20), (20, 60)]
mid = [np.sqrt(a * b) for a, b in bands]
rv = [np.median((S_v / S_u)[(omega >= a) & (omega < b)]) for a, b in bands]
rw = [np.median((S_w / S_u)[(omega >= a) & (omega < b)]) for a, b in bands]
plt.semilogx(omega[1:], S_v[1:] / S_u[1:], c="#BCE3D0", lw=0.3); plt.semilogx(omega[1:], S_w[1:] / S_u[1:], c="#E8C0C0", lw=0.3)
plt.semilogx(mid, rv, "o-", c="#0E8F5A", ms=8, label="S_v / S_u (band median)"); plt.semilogx(mid, rw, "s-", c="#990000", ms=8, label="S_w / S_u (band median)")
plt.axhline(4 / 3, c="k", ls="--", label="4/3"); plt.ylim(0, 2.5)
plt.xlabel("ω (rad/s)"); plt.ylabel("ratio"); plt.legend(); plt.show()
for (a, b), x, y in zip(bands, rv, rw):
    print(f"{a:>4}–{b:<3} rad/s: S_v/S_u = {x:.2f}, S_w/S_u = {y:.2f}")

<details><summary><b>Hint 1</b></summary>

Your `spectrum` function works for any component.

</details>
<details><summary><b>Hint 2</b></summary>

Use the same `K = 6` as for u, so the frequencies match.

</details>
<details><summary><b>Hint 3</b></summary>

`_, S_v = spectrum(d['v'], DT, K=6)` and the same for w.

</details>

---
### Your pair's answers (5') — for the report-out
1. Your slope, and the range of eddy sizes (m) where it holds
2. Where did the ratios reach 4/3, and where not? Why?

---
## Bonus (only if you finish early)
Set `case = "strong"` in the setup cell and rerun everything. Does the straight part move in ω, in metres, or both?
And the ratios of Task 4?